# Lab 6 — Fine-tuning a Vision Transformer on Florida Traffic Cameras

**AI for Mobility (CGN 6933) · Module 4 · reads with the Oct 1 session on model adaptation**

On Sept 10 you saw the **transfer dial**: a pretrained backbone, a task head, and a choice
about how much of the backbone you let move. You saw the four settings on a slide. This lab
turns the dial for real, on 2949 snapshots pulled from
494 FDOT traffic cameras around Tampa Bay, and measures what each
setting costs and buys.

The model is **ViT-Tiny** — the same vision transformer you met in the attention companion,
with the same 12 blocks and the same 196 patches, but 192 numbers per patch instead of 768.
That makes it **5,524,995 parameters**, about one fifteenth of ViT-Base, and
small enough that every experiment below runs on a laptop CPU in minutes.

| | |
|---|---|
| **What you build** | A 3-class traffic-density classifier on real Florida CCTV frames |
| **What it proves** | Where the accuracy in a fine-tune actually comes from — and how little of it survives sunset |
| **Check it against** | A 12-feature logistic regression, and the detector that wrote the labels |
| **Runs on** | Colab (CPU is fine, GPU is faster), macOS, Windows |
| **Needs** | `torch`, `timm`, `scikit-learn`, `matplotlib` · about 55 MB of downloads (31 MB of images, 22 MB of weights) |

---

### The honest disclosure, before anything else

The labels in this lab were **not written by a person**. They were written by a COCO-pretrained
Faster R-CNN — the same detector from the scooter case — counting vehicles in each frame. So
every accuracy number below is *agreement with a detector*, not agreement with the truth. That
is a real and common situation in transportation AI: the labels you can afford are the labels
some other model produced. Section 9 is about what that costs you.

## 0 · Setup

Three cells. Run them in order and move on.

In [ ]:
# ===========================================================================
#  CELL 1 — install. Safe to re-run; it skips what is already there.
# ===========================================================================
import importlib, subprocess, sys

def need(mod, pipname=None):
    try:
        importlib.import_module(mod); return None
    except ImportError:
        return pipname or mod

# timm 1.x is the pin that matters: the checkpoint names used below
# ("vit_tiny_patch16_224.augreg_in21k_ft_in1k") are timm-1.x style. On timm 0.x
# the same call raises "Unknown model", which is a confusing error to meet cold.
wanted = [need("torch"), need("torchvision"), need("timm", "timm>=1.0"),
          need("sklearn", "scikit-learn"), need("matplotlib"), need("PIL", "pillow")]
wanted = [w for w in wanted if w]
if wanted:
    print("installing:", wanted)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *wanted], check=True)
else:
    print("nothing to install")

import timm, torch
print("torch", torch.__version__, "· timm", timm.__version__,
      "· CUDA" if torch.cuda.is_available() else "· CPU only")

## 1 · 2949 frames off the wall of a traffic management center

FDOT publishes **4958** live CCTV cameras through FL511.
**717** of them sit inside a box drawn around Tampa Bay. None of
them come with labels. That is the normal condition of transportation video: enormous volume,
no annotation, and an operator watching a video wall deciding by eye which corridor needs
attention.

This dataset is one sweep of those cameras, repeated 6 times on
the evening of September 21 2026 — starting at
**17:53**
and ending at
**20:01**,
straight through sunset at 19:25. 2949 usable frames
from 494 cameras. **The same camera appears in every sweep**, which is
what makes the after-dark experiment in section 8 possible.

In [ ]:
# ===========================================================================
#  CELL 2 — get the data (about 30 MB) and read the label table.
# ===========================================================================
import os, zipfile, urllib.request, csv, numpy as np
from pathlib import Path

DATA_URL = "https://ai4mobility.github.io/_static/data/fl511_tampa_20260921.zip"
ZIP, ROOT = "fl511_tampa_20260921.zip", Path("fl511_tampa")

if not ROOT.exists():
    if not os.path.exists(ZIP):
        print(f"downloading {DATA_URL} (about 31 MB) ...")
        urllib.request.urlretrieve(DATA_URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)
    print("extracted to", ROOT.resolve())

rows = list(csv.DictReader(open(ROOT / "labels.csv")))
for r in rows:
    r["n_veh"] = int(r["n_veh"]); r["veh_area_frac"] = float(r["veh_area_frac"])
    r["src_w"] = int(r["src_w"]); r["src_h"] = int(r["src_h"])

print(f"{len(rows)} frames  ·  {len({r['camera_id'] for r in rows})} cameras  "
      f"·  {len({r['sweep'] for r in rows})} sweeps")
print("native resolutions present:",
      sorted({(r['src_w'], r['src_h']) for r in rows}, key=lambda t: -t[0]*t[1])[:6])

Look at the frames before you model them. This is not a tidy benchmark — it is what a
camera network actually returns.

In [ ]:
# ===========================================================================
#  CELL 3 — twelve frames, spread across the detector's vehicle count.
# ===========================================================================
import matplotlib.pyplot as plt
from PIL import Image

GREEN, DARK, MID, GOLD, SAND = "#006747", "#003E2A", "#2E7D5C", "#B79F4D", "#CFC493"

first = sorted({r["sweep"] for r in rows})[0]
day = sorted([r for r in rows if r["sweep"] == first], key=lambda r: r["n_veh"])
picks = [day[int(i * (len(day) - 1) / 11)] for i in range(12)]

fig, axes = plt.subplots(3, 4, figsize=(14, 7.4))
for ax, r in zip(axes.ravel(), picks):
    ax.imshow(Image.open(ROOT / r["path"])); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"camera {r['camera_id']} · detector says {r['n_veh']}",
                 fontsize=9, color=DARK)
fig.suptitle("FL511 Tampa Bay, one sweep, sorted by vehicle count", color=DARK, fontsize=13)
fig.tight_layout(); plt.show()

Three things are worth naming out loud before you go further, because each one comes
back later:

1. **Every frame carries burned-in text** — roadway, milepost, direction, sometimes a
   timestamp and a district logo. It is free metadata and a free shortcut. Section 9 checks
   which one the model used.
2. **The views are not comparable.** Some cameras look down a freeway for a mile, some look
   at one intersection approach, a few are pointed at a truck parking lot or, at this hour,
   at a wall of trees. A "vehicle count" means something different in each.
3. **The optics vary more than the traffic does.** Some feeds are sharp; some are hazed,
   defocused, rain-streaked, or compressed down to 352×240.

## 2 · Where the labels came from, and why that matters

Nobody counted these vehicles. A **torchvision `fasterrcnn_resnet50_fpn`** with COCO weights —
the detector from the scooter case — ran on every frame at its native resolution, kept
detections scoring ≥ 0.50 in the COCO classes *car, motorcycle, bus, truck*, and the count of
those boxes is `n_veh`.

Across the corpus: mean **7.19** vehicles per frame, median
**4**, max **58**, and
**19.0%** of frames with nothing detected at all.

We bin that into three classes, because an operator triaging a video wall wants a state, not a
number:

| class | rule | frames |
|---|---|---|
| **light (0-2)** | `n_veh` ≤ 2 | 1166 |
| **moderate (3-9)** | 3 ≤ `n_veh` ≤ 9 | 981 |
| **heavy (10+)** | `n_veh` ≥ 10 | 802 |

**This is weak supervision, and the rest of the lab is honest about it.** Every accuracy below
is *agreement with the detector*. Section 9 asks whether the detector deserved the trust.

## 3 · ViT-Tiny — the same transformer, one eighth as wide

You have already met this architecture twice. The tokens companion cut a frame into
**196 patches of 16×16
pixels** and showed that a patch is a choice of Δx, exactly like a cell length in the CTM. The
attention companion then opened ViT-Base and read its heads. ViT-Tiny is the same object with
one number changed: each patch becomes a vector of **192** numbers
instead of 768.

| | ViT-Tiny | ViT-Base |
|---|---|---|
| width (per-patch vector) | **192** | 768 |
| blocks · heads | 12 · 3 | 12 · 12 |
| patch projection | 147,648 | 590,592 |
| one transformer block | 444,864 | 7,087,872 |
| **total parameters** | **5,524,995** | 85,800,963 |
| 3-class head on `[CLS]` | **579** | 2307 |

Width is quadratic in the block: 768 ÷ 192
is 4, and 7,087,872 ÷ 444,864 is about 16. That one
ratio is why Tiny fits where Base does not.

Both are Apache-2.0 and both are one line of `timm`. Run the cell to see the dial you are about
to turn.

In [ ]:
# ===========================================================================
#  CELL 4 — the model, and the four settings of the transfer dial.
# ===========================================================================
import timm, torch, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

MODEL = "vit_tiny_patch16_224.augreg_in21k_ft_in1k"   # Apache-2.0, 5.5M parameters
NCLASS = 3

def make_model(nclass=NCLASS, name=MODEL):
    return timm.create_model(name, pretrained=True, num_classes=nclass)

def set_dial(m, dial):
    """dial = 0  -> head only          (the frozen probe)
       dial = 1  -> last block + head
       dial = 4  -> last 4 blocks + head
       dial = -1 -> everything          (full fine-tune)"""
    if dial < 0:
        for p in m.parameters(): p.requires_grad = True
    else:
        for p in m.parameters(): p.requires_grad = False
        for p in m.head.parameters(): p.requires_grad = True
        for p in m.norm.parameters(): p.requires_grad = True
        for b in m.blocks[len(m.blocks) - dial:]:
            for p in b.parameters(): p.requires_grad = True
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

m = make_model()
TOTAL = sum(p.numel() for p in m.parameters())
dials = [(0, "head only"), (1, "last block + head"), (4, "last 4 blocks + head"), (-1, "everything")]
counts = [(lab, set_dial(make_model(), d)) for d, lab in dials]

fig, ax = plt.subplots(figsize=(11, 3.2))
for i, (lab, c) in enumerate(counts):
    ax.barh(i, 100 * c / TOTAL, color=["#CFC493", "#B79F4D", "#2E7D5C", "#006747"][i], height=.62)
    ax.text(100 * c / TOTAL + 1.2, i, f"{c:,}  ({100*c/TOTAL:.2f}%)", va="center",
            fontsize=11, color="#003E2A")
ax.set_yticks(range(4)); ax.set_yticklabels([lab for lab, _ in counts], fontsize=11)
ax.set_xlim(0, 125); ax.set_xlabel("% of the 5,524,995 parameters that receive a gradient")
ax.set_title("The transfer dial, ViT-Tiny", color="#003E2A", fontsize=13, loc="left")
for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.tight_layout(); plt.show()

**963 parameters.** That is the whole of dial position 0 — a
192×3 matrix and three biases, sitting on top of
5,524,995 frozen ones. Sept 10 made the same point on ResNet-50 with the
number 4,098. The lesson survives the change of architecture: *training a model* and
*training a head* differ by four orders of magnitude, and most of the time nobody tells you
which one a paper did.

## 4 · The split, which is where most camera papers go wrong

494 cameras × 6 sweeps means every camera
contributes several frames, and consecutive frames from the same fixed camera are
near-duplicates — same pole, same lane markings, same billboard, same burned-in caption. Split
those at random and the model can memorise the view instead of reading the traffic, and the
test set will cheerfully confirm it.

So the split here is **camera-disjoint**: a camera is in train or in test, never both. Same
rule as the Sept 10 slides used for post-storm imagery — *split by corridor and by storm, never
by frame*.

There is a second rule, and it is the one that sets up section 8: **training uses daylight
sweeps only.** The after-dark frames are held out entirely.

| | frames | cameras |
|---|---|---|
| train — daylight, train cameras | **1375** | 346 |
| test — daylight, held-out cameras | **591** | 148 |
| test — after dark, held-out cameras | **296** | 148 |

Camera overlap between train and test: **0**.

In [ ]:
# ===========================================================================
#  CELL 5 — build the arrays and make the split. Nothing is trained yet.
# ===========================================================================
import random
from PIL import Image

SIZE, MEAN, STD = 224, np.float32([.5, .5, .5]), np.float32([.5, .5, .5])

def load_images(rows, crop_banner=False):
    out = []
    for r in rows:
        im = Image.open(ROOT / r["path"]).convert("RGB")
        if crop_banner:
            w, h = im.size
            im = im.crop((0, int(.06*h), w, int(.90*h))).resize((SIZE, SIZE), Image.BILINEAR)
        out.append(np.asarray(im, dtype=np.uint8))
    return np.stack(out)

rows = sorted(rows, key=lambda r: r["path"])
X    = load_images(rows)
cam  = np.array([r["camera_id"] for r in rows])
swp  = np.array([r["sweep"] for r in rows])
nveh = np.array([r["n_veh"] for r in rows])

y = np.zeros(len(rows), dtype=np.int64); y[nveh >= 3] = 1; y[nveh >= 10] = 2
CLASSES = ["light (0-2)", "moderate (3-9)", "heavy (10+)"]

SUNSET = "1925"                     # local sunset, Tampa, Sept 21 2026
is_day   = np.array([s.split("_")[1] <  SUNSET for s in swp])
is_night = ~is_day

u = sorted(set(cam.tolist())); random.Random(7).shuffle(u)
test_cams = set(u[:int(len(u) * .30)])
in_test   = np.array([c in test_cams for c in cam])

tr       = np.where(~in_test & is_day)[0]
te_day   = np.where( in_test & is_day)[0]
te_night = np.where( in_test & is_night)[0]

print(f"train (daylight)      {len(tr):5d} frames, {len(set(cam[tr]))} cameras")
print(f"test  (daylight)      {len(te_day):5d} frames, {len(set(cam[te_day]))} cameras")
print(f"test  (after dark)    {len(te_night):5d} frames")
print(f"camera overlap        {len(set(cam[tr]) & set(cam[te_day]))}")
print("class balance in train:", np.bincount(y[tr]))

## 5 · Two baselines, before any transformer

Never report a fine-tune without the two numbers it has to beat.

**Baseline 1 — call everything the most common class.** On daylight test frames that is
**35.5%**. Any model that lands near this has learned the
base rate and nothing else.

**Baseline 2 — twelve hand-written numbers and a logistic regression.** Mean brightness,
brightness spread, horizontal and vertical edge energy, the same three for the lower half of
the frame (where the pavement usually is), the three channel means, the 10th and 90th
brightness percentiles, and the fraction of very dark pixels. No learned features, no
pretraining, fits in **5.3 s**.

This is the baseline the room should respect, because it is what a consultant would try first
and it costs nothing to run on a Raspberry Pi.

In [ ]:
# ===========================================================================
#  CELL 6 — the two baselines.
# ===========================================================================
from sklearn.linear_model import LogisticRegression

maj = np.bincount(y[tr]).argmax()
print(f"majority class = {CLASSES[maj]!r}")
print(f"  daylight test   {(y[te_day]   == maj).mean():.4f}")
print(f"  after-dark test {(y[te_night] == maj).mean():.4f}")

def handfeat(Xu8, idx):
    out = []
    for k in idx:
        a = Xu8[k].astype(np.float32) / 255.; g = a.mean(2)
        out.append([g.mean(), g.std(),
                    np.abs(np.diff(g, axis=1)).mean(), np.abs(np.diff(g, axis=0)).mean(),
                    g[112:].mean(), g[112:].std(),
                    a[..., 0].mean(), a[..., 1].mean(), a[..., 2].mean(),
                    np.percentile(g, 10), np.percentile(g, 90), float((g < .15).mean())])
    return np.array(out, dtype=np.float32)

lr12 = LogisticRegression(max_iter=3000).fit(handfeat(X, tr), y[tr])
print(f"\n12 hand features + logistic regression")
print(f"  daylight test   {lr12.score(handfeat(X, te_day), y[te_day]):.4f}")
print(f"  after-dark test {lr12.score(handfeat(X, te_night), y[te_night]):.4f}")

> **Your turn — exercise 1.** Before running the next cell, write down what you expect
> the frozen ViT-Tiny probe to score, and why. If your reasoning is "ImageNet has a car class,
> so the features already know about cars," you are on the right track — hold onto that
> sentence, because section 7 will put it next to a case where it was badly wrong.

## 6 · One harness, four dial settings

The loop below is the whole of fine-tuning. Nothing about it is vision-specific — swap the
model and it is the BERT fine-tune from the crash-narrative work.

$$\mathcal{L} \;=\; -\log \operatorname{softmax}\!\big(W h_{[\mathrm{CLS}]} + b\big)_{y}$$

| symbol | what it is here |
|---|---|
| $h_{[\mathrm{CLS}]}$ | the 192-number summary vector the 12 blocks produce for the whole frame |
| $W, b$ | the head — a 192×3 matrix and 3 biases, 579 numbers, always trained |
| $y$ | the density class the detector assigned |
| the dial | *which other parameters also receive a gradient* |

**In words:** score the three classes from one summary vector, and push up the score of the
class the detector chose. The only decision that separates the four experiments is how far
back into the network that push is allowed to travel.

The learning rate is 5e-5 with cosine decay, batch 16, 3 epochs, AdamW. Small learning rates
are not a stylistic preference: the backbone already encodes something worth keeping, and a
large step erases it.

In [ ]:
# ===========================================================================
#  CELL 7 — the training loop. Runs on CPU; a GPU just makes it faster.
# ===========================================================================
import time

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(2)

def to_t(Xu8, idx):
    a = (Xu8[idx].astype(np.float32) / 255. - MEAN) / STD
    return torch.from_numpy(a.transpose(0, 3, 1, 2)).to(DEV)

def evaluate(m, Xu8, idx, bs=32):
    m.eval(); c = 0
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            c += (m(to_t(Xu8, idx[i:i+bs])).argmax(1).cpu().numpy() == y[idx[i:i+bs]]).sum()
    return c / len(idx)

def run_dial(dial, Xu8=None, epochs=3, bs=16, lr=5e-5, seed=0, quiet=False):
    Xu8 = X if Xu8 is None else Xu8
    torch.manual_seed(seed)
    m = make_model().to(DEV)
    n_train = set_dial(m, dial)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=lr, weight_decay=.01)
    lossf = torch.nn.CrossEntropyLoss()
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs * ((len(tr) + bs - 1) // bs))
    t0 = time.time()
    for ep in range(epochs):
        m.train(); perm = np.random.RandomState(seed + ep).permutation(tr); tot = 0.
        for i in range(0, len(perm), bs):
            j = perm[i:i+bs]
            l = lossf(m(to_t(Xu8, j)), torch.from_numpy(y[j]).to(DEV))
            l.backward(); opt.step(); opt.zero_grad(); sched.step(); tot += l.item() * len(j)
        if not quiet:
            print(f"   epoch {ep+1}  loss {tot/len(perm):.4f}   {time.time()-t0:.0f}s")
    return dict(model=m, trainable=n_train, seconds=time.time() - t0,
                day=evaluate(m, Xu8, te_day), night=evaluate(m, Xu8, te_night))

## 7 · Turning the dial

Four runs, identical in every respect except how far back the gradient is allowed to travel.
Each is 3 epochs, batch 16, learning rate 5e-5, cosine decay.

In [ ]:
# ===========================================================================
#  CELL 8 — the four runs. On 2 CPU threads this is about six minutes total.
# ===========================================================================
results = {}
for dial, label in [(0, "head only"), (1, "last block + head"),
                    (4, "last 4 blocks + head"), (-1, "everything")]:
    print(f"\ndial = {label}")
    results[label] = run_dial(dial)
    r = results[label]
    print(f"   trainable {r['trainable']:,}   daylight {r['day']:.4f}   after dark {r['night']:.4f}")

Three of those four lines behave the way the Sept 10 slide promised. One does not.

| dial | trainable | daylight test | after-dark test |
|---|---|---|---|
| *majority class* | 0 | 35.5% | 13.9% |
| *12 hand features* | 12 | 37.7% | 31.8% |
| head only | 963 | **34.5%** | 25.7% |
| last block + head | 445,827 | 51.4% | 38.5% |
| last 4 blocks + head | 1,780,419 | 64.0% | 55.7% |
| everything | 5,524,995 | **71.2%** | 67.6% |

**Head only scores 34.5% — below the majority class, below twelve
hand-written numbers.** Read literally, that says a ViT-Tiny pretrained on 14 million images
carries no usable information about how many vehicles are in a traffic-camera frame. That
conclusion would be wrong, and the next cell shows why.

In [ ]:
# ===========================================================================
#  CELL 9 — the same frozen features, fit properly.
#  Nothing changes about the backbone. Only the thing on top of it changes.
# ===========================================================================
backbone = timm.create_model(MODEL, pretrained=True, num_classes=0).to(DEV).eval()

def features(idx, bs=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            out.append(backbone(to_t(X, idx[i:i+bs])).cpu().numpy())
    return np.concatenate(out)

Ftr, Fday, Fnight = features(tr), features(te_day), features(te_night)
probe = LogisticRegression(max_iter=5000).fit(Ftr, y[tr])
print(f"frozen features + logistic regression   ({Ftr.shape[1]}-d, "
      f"{Ftr.shape[1]*3+3} trainable numbers)")
print(f"  daylight test   {probe.score(Fday, y[te_day]):.4f}")
print(f"  after-dark test {probe.score(Fnight, y[te_night]):.4f}")

**56.5%.** Same frozen backbone, same
579 numbers on top, more than twenty points better than the head-only
fine-tune run — because a learning rate of 5e-5 for three epochs is nowhere near enough to fit
a linear layer from scratch. Re-run dial 0 at 1e-3 and it recovers to
57.0%.

**The dial position and the optimizer are not independent choices.** Freezing the backbone
changes what the trainable part *is* — a fresh linear layer instead of a nudge to a trained
network — and the recipe has to change with it. This is the single most common way a "we tried
freezing and it didn't work" result gets written down, and it is worth more to you than any
accuracy number in this notebook.

Corrected, the daylight table reads:

| what was trained | trainable | daylight |
|---|---|---|
| nothing (majority class) | 0 | 35.5% |
| 12 hand-written features | 12 | 37.7% |
| head only, **logistic regression to convergence** | 579 | 56.5% |
| head only, 3 epochs at **1e-3** | 963 | 57.0% |
| head only, 3 epochs at 5e-5 | 963 | 34.5% ← undertrained |
| last block + head, 3 epochs at 5e-5 | 445,827 | 51.4% ← also undertrained |
| last 4 blocks + head | 1,780,419 | 64.0% |
| everything | 5,524,995 | **71.2%** |

Two rows sit below where they belong, and it is the same artifact both times: the fewer
parameters you unfreeze, the more of the work falls on parameters that started from random
noise, and the longer or faster they need to be trained. The two rows to quote are the
converged probe at **56.5%** and the full fine-tune at
**71.2%**.

Read that way: about
19
points come free from features somebody else paid for, and another
15 have to
be bought by letting the backbone move.

> **Your turn — exercise 2.** `run_dial(2)` and `run_dial(8)` are one line each. Add
> them to the table. Is the curve from 0 to 12 blocks smooth, or is there a block where most
> of the gain appears? Yosinski et al. (NeurIPS 2014) predicted a shape for this — say whether
> yours matches.

## 8 · After dark, where both models stop working

Training used daylight frames only. The held-out cameras were also photographed at
19:35
(dusk) and
20:01
(dark), and those frames were never trained on. The full fine-tune scores
**71.2%** in daylight and
**67.6%** after sunset.

A four-point drop across a lighting change that destroys most detectors looks too good. It is.
Run the next cell before you believe it.

In [ ]:
# ===========================================================================
#  CELL 10 — what the TEACHER did as the sun went down.
# ===========================================================================
sweeps = sorted(set(swp))
mean_n  = [nveh[swp == s].mean()          for s in sweeps]
zero_f  = [(nveh[swp == s] == 0).mean()   for s in sweeps]
bright  = [X[swp == s].astype(np.float32).mean() / 255. for s in sweeps]
labels  = [f"{s.split('_')[1][:2]}:{s.split('_')[1][2:]}" for s in sweeps]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(labels, mean_n, "o-", color=GREEN, lw=2)
ax[0].set_ylabel("mean vehicles found per frame"); ax[0].set_ylim(0, None)
ax[0].axvline(3.5, color=GOLD, ls="--"); ax[0].text(3.55, max(mean_n)*.9, " sunset", color=DARK)
ax[1].plot(labels, [100*z for z in zero_f], "o-", color=GREEN, lw=2, label="frames with 0 detections (%)")
ax[1].plot(labels, [100*b for b in bright], "s--", color=MID, lw=1.6, label="mean brightness (%)")
ax[1].axvline(3.5, color=GOLD, ls="--"); ax[1].legend(fontsize=9)
for a in ax:
    a.set_title("", color=DARK)
    for s in ("top", "right"): a.spines[s].set_visible(False)
fig.suptitle("The label source, sweep by sweep", color=DARK, fontsize=13)
fig.tight_layout(); plt.show()

for s, m_, z in zip(labels, mean_n, zero_f):
    print(f"  {s}   mean {m_:5.2f} vehicles   {100*z:5.1f}% of frames empty")

The detector's mean count falls from **10.33**
vehicles per frame at 17:53
to **2.13** at
20:01 — a
79% collapse
in about two hours. The share of frames where it finds nothing at all goes from
**11.2%** to
**40.4%**.

Evening traffic in Tampa does *not* fall by that much between 6 PM and 8 PM. What collapsed is
the detector.

So the after-dark accuracy of **67.6%** is not evidence that
the model handles darkness. It is evidence that **the student learned to agree with the teacher,
including where the teacher is wrong.** After sunset the labels drift toward "light", the model
drifts with them, and agreement stays high while both drift away from the road.

*This dataset cannot separate "traffic genuinely thinned out" from "the detector went blind."
Exercise 4 in section 11 asks what measurement would.*

## 9 · Is it reading the road, or the caption?

Every frame has the roadway name, milepost and direction burned into it. A model that learned
"I-275 at MM 45 is always heavy" would score well on the training cameras — but the test
cameras are different cameras with different captions, so that particular shortcut cannot
survive a camera-disjoint split. A subtler one can: the *style* of the banner identifies the
district, and districts differ in how busy their roads are.

The test is blunt. Cut the banner off and retrain.

In [ ]:
# ===========================================================================
#  CELL 11 — same everything, with the top 6% and bottom 10% of each frame gone.
# ===========================================================================
Xc = load_images(rows, crop_banner=True)

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax, k in zip(axes, [tr[0], tr[1], te_day[0], te_day[1]]):
    ax.imshow(np.concatenate([X[k], Xc[k]], axis=1)); ax.set_xticks([]); ax.set_yticks([])
axes[0].set_ylabel("full | cropped", fontsize=9, color=DARK)
fig.suptitle("what the crop removes", color=DARK); fig.tight_layout(); plt.show()

cropped = run_dial(-1, Xu8=Xc)
print(f"\nfull frame  daylight {results['everything']['day']:.4f}   "
      f"after dark {results['everything']['night']:.4f}")
print(f"banner gone daylight {cropped['day']:.4f}   after dark {cropped['night']:.4f}")

**69.2% against 71.2%.** Losing
the banner — and with it about 16% of every frame, including some road — costs about
2 points.
The model was not mainly reading the caption.

That is the answer you want, and it is worth noticing that you only get to say it because you
ran the experiment. "The model probably isn't cheating" is not a finding; a two-point delta is.

There is a second, less comfortable audit. The teacher under-counts on bad optics, and the
student inherits that too:

In [ ]:
# ===========================================================================
#  CELL 12 — does the label source see less on blurrier cameras? (daylight only)
# ===========================================================================
g = X[is_day].astype(np.float32).mean(3) / 255.
sharp = np.array([np.abs(np.diff(im, axis=1)).mean() + np.abs(np.diff(im, axis=0)).mean()
                  for im in g])
nv = nveh[is_day]
q1, q3 = np.quantile(sharp, [.25, .75])
print(f"blurriest quarter of daylight frames: {nv[sharp <= q1].mean():.2f} vehicles found")
print(f"sharpest  quarter of daylight frames: {nv[sharp >= q3].mean():.2f} vehicles found")
print(f"correlation(sharpness, count) = {np.corrcoef(sharp, nv)[0,1]:.3f}")

**6.32 against
11.27 vehicles**, in daylight, so darkness is not the
explanation. A hazed or defocused camera is labelled *light* whatever is on the road, and the
model learns that haze means light traffic. In an operations setting that is exactly backwards:
the cameras you most want flagged are often the ones nobody has cleaned.

## 10 · What it costs, and whether the big model was worth it

Everything above used ViT-Tiny. The obvious question is what 85,800,963
parameters would have bought instead of 5,524,995. Two answers, and the second
one is the one that decides deployments.

In [ ]:
# ===========================================================================
#  CELL 13 — timing on whatever machine you are on, both models.
# ===========================================================================
def bench(name):
    m = timm.create_model(name, pretrained=True, num_classes=NCLASS).to(DEV)
    xb, yb = to_t(X, np.arange(8)), torch.from_numpy(y[:8]).to(DEV)
    m.eval()
    with torch.no_grad(): m(xb[:2])
    t0 = time.time()
    with torch.no_grad(): m(xb)
    fwd = (time.time() - t0) / 8
    m.train(); opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    lf = torch.nn.CrossEntropyLoss()
    lf(m(xb[:2]), yb[:2]).backward(); opt.zero_grad()
    t0 = time.time(); lf(m(xb), yb).backward(); opt.step(); opt.zero_grad()
    step = (time.time() - t0) / 8
    n_p = sum(p.numel() for p in m.parameters())
    print(f"{name.split('.')[0]:28s} {n_p:>11,} params   "
          f"forward {1000*fwd:6.1f} ms/frame ({1/fwd:5.1f} fps)   "
          f"train step {1000*step:6.1f} ms/frame")
    del m, opt

for nm in ["vit_tiny_patch16_224.augreg_in21k_ft_in1k",
           "vit_base_patch16_224.augreg_in21k_ft_in1k"]:
    bench(nm)

On 2 CPU threads:

| | ViT-Tiny | ViT-Base | ratio |
|---|---|---|---|
| parameters | 5,524,995 | 85,800,963 | 15.5× |
| forward, one frame | **13.6 ms** (73.6 fps) | 143.0 ms (7.0 fps) | 10.5× |
| one training step | 48.5 ms | 483.4 ms | 10.0× |
| frozen probe, daylight | 56.5% | 58.0% | |
| feature extraction, 2949 frames | 38.9 s | 356.5 s | |

**The ViT-Base fine-tune is not in that table, and the reason is worth more than
the number would have been.** It was killed part-way through its third epoch: at batch 16 the
weights, the AdamW moments and the activations did not fit in 8 GB of RAM. That is the free
Colab tier, and it is also most laptops. The
10×
step time was the visible cost; the memory ceiling was the one that actually stopped the run.

What did survive is the comparison that needs no training at all: as a **frozen** feature
extractor, ViT-Base scores 58.0% against ViT-Tiny's
56.5% — about a point and a half, for
16× the parameters
and 9× the
time to extract.

**Why that matters for the pole, not the laptop.** 73.6 frames per
second on two general-purpose CPU threads means one ViT-Tiny can keep up with a live camera
with no accelerator at all — and the Raspberry Pi 5 in the course inventory has no
accelerator. The Jetson Orin Nano is not needed to run *this*; it is needed to run several of
these at once, or one of them at 30 fps alongside a detector. Sizing that honestly is the
Oct 22 lab, and the number you carry into it is
**13.6 ms per frame at 224×224 on two CPU threads**.

**And the cost upstream.** Labelling these 2949 frames with
`fasterrcnn_resnet50_fpn` took **1.041 s per frame** on the same two
threads — 77×
the cost of running the finished classifier. That is the trade this whole lab is really about:
**pay the big model once, offline, to teach a small model you can afford to run forever.**

## 11 · What this was for

```
Fit card — Fine-tuning a pretrained vision backbone

Idea in one sentence:  Keep somebody else's visual features, replace the last layer with
                       your task, and choose how much of the backbone you let move.

Why it works (the logic):  The early blocks learned edges, texture and object parts from
                       millions of photographs; a traffic camera is still photographs, so
                       those blocks transfer. Only the last layers are specific to
                       "which of 1,000 ImageNet nouns is this", and those are the ones
                       worth replacing.

Fits these tasks:  classification · extraction — a fixed camera or sensor producing images,
                       a small label budget, and a decision with a handful of states
                       (density level, wet/dry, sign type, damage/no damage, camera OK/blocked).

Needs this data:  a few hundred to a few thousand labelled images per class · labels can be
                       weak (another model's output) if you measure the cost · whoever owns
                       the cameras owns the data, which for CCTV is the DOT.

Breaks when:  the test images come from a condition the training images never had —
                       after dark, heavy rain, a re-aimed camera, a different vendor's optics.
                       It fails confidently, not loudly.

Keep the classical method when:  the signal is a few global statistics (brightness, edge
                       energy, occupancy), the deployment has no accelerator, or you need an
                       auditable rule. A 12-feature logistic regression that a reviewer can
                       read is often worth more than four points of accuracy.
```

### Where this goes next, in the field

| Domain | Task | Data you would need | The baseline to beat |
|---|---|---|---|
| **Traffic operations (TSM&O)** | Triage a video wall: which of 700 cameras is showing a queue right now | the DOT's own CCTV archive + a few thousand operator-confirmed labels | the operator, and the loop/radar detectors already on that corridor |
| **Asset management** | Camera-health triage — obstructed, defocused, mis-aimed, dark | the same frames, labelled by a technician once | the maintenance ticket that gets filed when someone notices |
| **Road safety** | Wet-pavement and standing-water flags after a storm | camera frames paired with RWIS or rain-gauge records | the weather feed, which is right most of the time and blind to local ponding |
| **Edge AI (Oct 22)** | Run the classifier on the pole instead of shipping video | the fine-tuned weights, quantised | shipping the video to a central detector, which is what happens today |

### Evaluation close

- **Baseline.** Majority class 35.5%, twelve hand features
  37.7%. Report both or the fine-tune number means
  nothing.
- **The split and why it is honest.** Camera-disjoint, so no view appears in both halves;
  daylight-only training, so the after-dark set is a real distribution shift and not a
  shuffled copy of the training set.
- **The ugly cases.** Hazed and defocused cameras, where the teacher under-counts;
  truck-parking and construction-yard views, where "vehicles present" does not mean traffic;
  cameras pointed at trees; and every frame after sunset, where the teacher stops working
  before the student does.

### Check yourself

1. A district wants this running on all 717 Tampa Bay cameras,
   updating every minute. Using the timing in section 10, what hardware do you ask for, and
   what would you change about the model first — dial position, input size, or both?
2. Your fine-tune reaches 71.2% agreement with the detector.
   The district asks "so it is 71.2% accurate?" Write the two
   sentences you say back.
3. Someone proposes re-labelling 300 frames by hand instead of trusting the detector. Which
   300 would you pick, and what would you do with the result — retrain, re-weight, or only
   report it?
4. The night result in section 8 has two possible explanations that this dataset cannot
   separate. Name them, and name the one extra measurement that would separate them.

---

*Built with `vit_tiny_patch16_224.augreg_in21k_ft_in1k` (Apache-2.0) and
`fasterrcnn_resnet50_fpn` COCO weights. Images are public FDOT FL511 camera feeds collected
2026-09-21. Every number in this notebook is produced by the cells above; the reproducer lives
in `Labs/vit_finetune/`.*